**Price Gap Moments Analysis**

This notebook reads in price gap, tariff, and gravity data for multiple years, combines them, and reports correlations between key variables.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
# Define years and read in data
years = ["2004", "2011", "2017"]
all_dfs = []

# Read gravity data (same for all years)
grav_df = pd.read_csv("../data/top30_gravity_data.csv")

for year in years:
    # Read price gap data
    df = pd.read_csv(f"../data/pricegap-df-{year}.csv")
    df = df.rename(columns={"exporter": "iso_o", "importer": "iso_d"})
    
    # Read tariff data
    tariffs_df = pd.read_csv(f"../data/tariffs-{year}.csv")
    tariffs_df = tariffs_df.rename(columns={"exporter": "iso_o", "importer": "iso_d"})
    
    # Merge price gap with tariffs
    df = df.merge(tariffs_df, on=["iso_o", "iso_d"], how="inner")
    
    # Merge with gravity data
    df = df.merge(grav_df, on=["iso_o", "iso_d"], how="inner")
    
    # Add year column
    df["year"] = year
    
    all_dfs.append(df)
    print(f"Year {year}: {len(df)} observations")

# Combine all years
big_df = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal observations (all years): {len(big_df)}")

Year 2004: 870 observations
Year 2011: 870 observations
Year 2017: 870 observations

Total observations (all years): 2610


In [3]:
# Filter out extreme trade share values (Xni ≈ 0 or Xni ≈ 1)
big_df = big_df[~np.isclose(big_df["Xni"], 1.0)]
big_df = big_df[~np.isclose(big_df["Xni"], 0.0)]

print(f"Observations after filtering: {len(big_df)}")
big_df.head()

Observations after filtering: 2604


,iso_o,iso_d,E_importer,Xni,logXni,dni,dni2,dni3,τni,year,...,dist,importer,exporter,distbin,bin375,bin750,bin1500,bin3000,bin6000,binmax
0,AUT,AUS,2.356209e+11,0.004297,-5.449770,0.874112,0.480179,0.307567,0.339296,2004,...,9937.199,1,2,6.0,0,0,0,0,0,1
1,BEL,AUS,2.356209e+11,0.014961,-4.202340,0.823992,0.381552,0.294151,0.287386,2004,...,10416.159,1,3,6.0,0,0,0,0,0,1
2,BRA,AUS,2.356209e+11,0.001459,-6.529799,0.654726,0.459052,0.316882,0.110739,2004,...,8310.602,1,4,6.0,0,0,0,0,0,1
3,CAN,AUS,2.356209e+11,0.008851,-4.727270,0.488547,0.167467,0.120297,0.293881,2004,...,9687.172,1,5,6.0,0,0,0,0,0,1
4,CHN,AUS,2.356209e+11,0.059249,-2.826004,0.718615,0.455234,0.203520,0.175503,2004,...,5566.461,1,6,5.0,0,0,0,0,1,0


In [5]:
# Function to compute correlation with confidence interval
def correlation_with_ci(x, y, name_x, name_y, alpha=0.10):
    """Compute Pearson correlation with confidence interval."""
    # Remove any NaN/Inf values
    mask = np.isfinite(x) & np.isfinite(y)
    x_clean = x[mask]
    y_clean = y[mask]
    
    n = len(x_clean)
    r, p_value = stats.pearsonr(x_clean, y_clean)
    
    # Fisher z-transformation for confidence interval
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    z_crit = stats.norm.ppf(1 - alpha/2)
    z_lo, z_hi = z - z_crit * se, z + z_crit * se
    ci_lo, ci_hi = np.tanh(z_lo), np.tanh(z_hi)
    
    print(f"Correlation of {name_x} and {name_y}")
    print(f"  Pearson r = {r:.4f}")
    print(f"  p-value = {p_value:.4e}")
    print(f"  n = {n}")
    print(f"  {int((1-alpha)*100)}% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    print()

---
### Summary Statistics

In [13]:
# Summary statistics for dni by year
dni_summary = []

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    dni_summary.append({
        "Year": year,
        "N": len(df_subset),
        "Mean": df_subset["dni"].mean(),
        "Median": df_subset["dni"].median(),
        "Min": df_subset["dni"].min(),
        "Max": df_subset["dni"].max(),
        "Std": df_subset["dni"].std()
    })

dni_stats_df = pd.DataFrame(dni_summary)
print("Summary Statistics for dni")
dni_stats_df

Summary Statistics for dni


,Year,N,Mean,Median,Min,Max,Std
0,All,2604,0.94,0.86,0.19,3.13,0.42
1,2004,866,0.92,0.90,0.25,2.27,0.32
2,2011,868,0.98,0.83,0.19,3.13,0.51
3,2017,870,0.91,0.84,0.21,2.26,0.39


In [14]:
# Summary statistics for dni2 by year
dni2_summary = []

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    dni2_summary.append({
        "Year": year,
        "N": len(df_subset),
        "Mean": df_subset["dni2"].mean(),
        "Median": df_subset["dni2"].median(),
        "Min": df_subset["dni2"].min(),
        "Max": df_subset["dni2"].max(),
        "Std": df_subset["dni2"].std()
    })

dni2_stats_df = pd.DataFrame(dni2_summary)
print("Summary Statistics for dni2")
dni2_stats_df

Summary Statistics for dni2


,Year,N,Mean,Median,Min,Max,Std
0,All,2604,0.34,0.34,0.08,0.77,0.13
1,2004,866,0.37,0.38,0.10,0.76,0.12
2,2011,868,0.34,0.33,0.08,0.73,0.13
3,2017,870,0.32,0.31,0.08,0.77,0.13


---
### Summary Correlation Table

In [19]:
# Build a summary table of correlations by year (using log(dni))
summary_data = []

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    # Calculate correlations with p-values using log(dni)
    r_dist, p_dist = stats.pearsonr(np.log(df_subset["dist"]), np.log(df_subset["dni"]))
    r_border, p_border = stats.pearsonr(df_subset["border"], np.log(df_subset["dni"]))
    r_tariff, p_tariff = stats.pearsonr(np.log(1.0 + 0.01 * df_subset["tariff"]), np.log(df_subset["dni"]))
    
    summary_data.append({
        "Year": year,
        "N": len(df_subset),
        "Corr(log(dni), log(dist))": r_dist,
        "p-value (dist)": p_dist,
        "Corr(log(dni), border)": r_border,
        "p-value (border)": p_border,
        "Corr(log(dni), log(1+tariff))": r_tariff,
        "p-value (tariff)": p_tariff
    })

summary_df = pd.DataFrame(summary_data)
pd.set_option('display.float_format', '{:.4f}'.format)
summary_df

,Year,N,"Corr(log(dni), log(dist))",p-value (dist),"Corr(log(dni), border)",p-value (border),"Corr(log(dni), log(1+tariff))",p-value (tariff)
0,All,2604,0.4854,0.0000,-0.2323,0.0000,0.3498,0.0000
1,2004,866,0.3083,0.0000,-0.1226,0.0003,0.2645,0.0000
2,2011,868,0.5624,0.0000,-0.2632,0.0000,0.4250,0.0000
3,2017,870,0.5579,0.0000,-0.2944,0.0000,0.4298,0.0000


In [20]:
# Build a summary table of correlations by year using log(dni2)
summary_data_dni2 = []

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    # Calculate correlations with p-values using log(dni2)
    r_dist, p_dist = stats.pearsonr(np.log(df_subset["dist"]), np.log(df_subset["dni2"]))
    r_border, p_border = stats.pearsonr(df_subset["border"], np.log(df_subset["dni2"]))
    r_tariff, p_tariff = stats.pearsonr(np.log(1.0 + 0.01 * df_subset["tariff"]), np.log(df_subset["dni2"]))
    
    summary_data_dni2.append({
        "Year": year,
        "N": len(df_subset),
        "Corr(log(dni2), log(dist))": r_dist,
        "p-value (dist)": p_dist,
        "Corr(log(dni2), border)": r_border,
        "p-value (border)": p_border,
        "Corr(log(dni2), log(1+tariff))": r_tariff,
        "p-value (tariff)": p_tariff
    })

summary_df_dni2 = pd.DataFrame(summary_data_dni2)
summary_df_dni2

,Year,N,"Corr(log(dni2), log(dist))",p-value (dist),"Corr(log(dni2), border)",p-value (border),"Corr(log(dni2), log(1+tariff))",p-value (tariff)
0,All,2604,0.5365,0.0000,-0.2406,0.0000,0.3592,0.0000
1,2004,866,0.3838,0.0000,-0.1584,0.0000,0.2444,0.0000
2,2011,868,0.6539,0.0000,-0.3086,0.0000,0.4330,0.0000
3,2017,870,0.5749,0.0000,-0.2545,0.0000,0.4226,0.0000
